# Canwell Glacier HRRR-AK, WY2020–WY2026

Inspect the combined hourly forecast dataset, check coverage and replacement leads, and
compare modeled weather at the grid cells nearest the five Canwell sites.

In [ ]:
from pathlib import Path
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data" / "canwell_readme.txt").is_file())
path = Path(os.environ.get("CANWELL_NETCDF",
                           root / "data" / "hrrrak_canwell_f567_WY2020-WY2026.nc"))
if not path.is_file():
    raise FileNotFoundError(f"{path}. Run scripts/process_hrrrak.py from the repo root.")
ds = xr.open_dataset(path, engine="netcdf4", decode_timedelta=False)
ds

## Hourly coverage and source forecasts

`time` is forecast valid time in UTC. Folder water years follow the **requested initialization
date**. WY2020 includes the August–September 2019 spin-up, and September 30
initializations can verify on October 1. Replacement forecasts have shorter actual leads
and the same requested valid hour.

In [ ]:
time = ds.indexes["time"]
assert time.is_unique and time.is_monotonic_increasing
expected = pd.date_range(time[0], time[-1], freq="h")
missing = expected.difference(time)
actual_lead = pd.to_timedelta(ds.forecast_hour.values, unit="h")
assert (pd.DatetimeIndex(ds.forecast_reference_time.values) + actual_lead).equals(time)
replacement = ds.forecast_hour.values != ds.requested_forecast_hour.values

print(f"{len(time):,} valid hours from {time[0]} to {time[-1]} UTC")
print(f"{len(missing):,} missing hours within this span")
if len(missing):
    display(missing[:20])

coverage = pd.DataFrame({
    "Source water year": ds.source_water_year.values,
    "Valid time (UTC)": time,
    "Replacement": replacement,
}).groupby("Source water year").agg(
    Hours=("Valid time (UTC)", "size"),
    First_valid_UTC=("Valid time (UTC)", "min"),
    Last_valid_UTC=("Valid time (UTC)", "max"),
    Replacements=("Replacement", "sum"),
)
display(coverage)

## Native grid and nearest model cells

The table pairs each station with its nearest HRRR-AK cell center. Grid geopotential
height can differ from the station elevation, especially over steep terrain.

In [ ]:
site_text = (root / "data" / "canwell_readme.txt").read_text()
sites = []
for block in site_text.split("### "):
    match = re.search(r"DD: ([0-9.]+),\s*(-[0-9.]+)", block)
    if match:
        name = block.split(" ###")[0].strip()
        lat, lon = map(float, match.groups())
        elevation = float(re.search(r"Elev\.: ([0-9.]+)m", block)[1])
        sites.append((name, lat, lon, elevation))

grid_lat = ds.latitude.values
grid_lon = ds.longitude.values
rows = []
for name, lat, lon, elevation in sites:
    haversine = (np.sin(np.deg2rad(grid_lat - lat) / 2)**2
                 + np.cos(np.deg2rad(lat)) * np.cos(np.deg2rad(grid_lat))
                 * np.sin(np.deg2rad(grid_lon - lon) / 2)**2)
    y, x = np.unravel_index(haversine.argmin(), grid_lat.shape)
    distance_km = 2 * 6371.229 * np.arcsin(np.sqrt(haversine[y, x]))
    rows.append((name, lat, lon, elevation, y, x, distance_km))
cells = pd.DataFrame(rows, columns=["Site", "Latitude", "Longitude", "Station elevation (m)",
                                    "y index", "x index", "Distance to cell center (km)"])
point = ds.isel(y=xr.DataArray(cells["y index"].values, dims="site"),
                x=xr.DataArray(cells["x index"].values, dims="site"))
cells["Model geopotential height (m)"] = point.orog.isel(time=0).values
display(cells[["Site", "Station elevation (m)", "Model geopotential height (m)",
               "y index", "x index", "Distance to cell center (km)"]].round(2))

fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
mesh = ax.pcolormesh(grid_lon, grid_lat, ds.orog.isel(time=0), shading="nearest", cmap="terrain")
fig.colorbar(mesh, ax=ax, label="Model geopotential height (m)")
ax.scatter(point.longitude, point.latitude, facecolors="none", edgecolors="white",
           s=90, label="Nearest HRRR-AK cells")
ax.scatter(cells.Longitude, cells.Latitude, s=22, c="black", label="Canwell sites")
ax.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)",
       aspect=1 / np.cos(np.deg2rad(grid_lat.mean())))
ax.legend(loc="upper left")
plt.show()

## Hourly weather at Look Eyrie AWS

The temperature, wind and precipitation come from the nearest model cell. Daily precipitation
sums `precip_hourly` across 24 one-hour intervals ending at midnight UTC. The variable
`precip_total` accumulates from each forecast's actual initialization and has overlapping
windows across consecutive forecast hours.

In [ ]:
site = cells.index[cells.Site.eq("Look Eyrie AWS")][0]
start, end = "2025-09-01", "2025-10-01"
sample = point[["temp", "precip_hourly", "wind"]].isel(site=site).sel(time=slice(start, end)).load()
# Bin interval-ending timestamps into the UTC day where the precipitation fell.
daily_precip = sample.precip_hourly.resample(time="1D", closed="right").sum(
    skipna=True, min_count=24
).dropna("time")

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7), layout="constrained")
axes[0].plot(sample.time, sample.temp - 273.15)
axes[0].set(ylabel="2 m air temp. (°C)", title="Look Eyrie AWS nearest HRRR-AK cell")
axes[1].bar(daily_precip.time.values, daily_precip.values, align="edge", width=0.85)
axes[1].set(ylabel="Daily precip. (mm)")
axes[2].plot(sample.time, sample.wind)
axes[2].set(ylabel="10 m wind (m/s)", xlabel="UTC")
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()